In [1]:
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder

import ecgformer

In [7]:
# beats_train_x = np.load('/home/matrioszka/ptb-xl/X_train.npy')
# beats_train_y = np.load('/home/matrioszka/ptb-xl/y_train.npy', allow_pickle=True)
# beats_test_x = np.load('/home/matrioszka/ptb-xl/X_test.npy')
# beats_test_y = np.load('/home/matrioszka/ptb-xl/y_test.npy', allow_pickle=True)
beats_x = np.load('/home/matrioszka/mit-bih/mitbih_beats_x.npy', allow_pickle=True)
beats_y = np.load('/home/matrioszka/mit-bih/mitbih_beats_y.npy', allow_pickle=True)
print(beats_x.shape, beats_y.shape)

(85243, 150) (85243,)


In [12]:
beats_y = np.array(['N' if label=='N' else 'O' for label in beats_y])
# split into train/test
from sklearn.model_selection import train_test_split
beats_train_x, beats_test_x, beats_train_y, beats_test_y = train_test_split(
    beats_x, beats_y, test_size=0.2, random_state=45, stratify=beats_y)

le = LabelEncoder()
y_train = le.fit_transform(beats_train_y)
y_test  = le.transform(beats_test_y)

print(le.classes_)

X_train_t = torch.tensor(beats_train_x, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_test_t = torch.tensor(beats_test_x, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# Wrap in TensorDataset
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)

# Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

class_counts = np.bincount(y_train)  # assumes integer-encoded labels 0–4
print(class_counts)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()  # normalize to sum to 1
print(class_weights)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to("cuda")
criterion = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

['N' 'O']
[59366  8828]
[0.1294542 0.8705458]


In [14]:
model     = ecgformer.build_model(input_length=150, patch_size=10, num_classes=2, device="cuda")
optimizer = ecgformer.build_optimizer(model)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=1e-3,
        steps_per_epoch=len(train_loader),
        epochs=100,
        pct_start=0.1,
    )

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ECGformer | trainable params: {total_params:,}")

best_val_loss = float("inf")
epochs = 100

for epoch in range(1, epochs):
    train_loss = ecgformer.train_one_epoch(model, train_loader, optimizer, scheduler, "cuda", criterion)
    val_metrics = ecgformer.evaluate(model, test_loader, "cuda", criterion)
    val_loss, val_acc = val_metrics["loss"], val_metrics["accuracy"]

    scheduler.step(val_loss)

    print(
        f"Epoch [{epoch:3d}/{epochs}] "
        f"train_loss={train_loss:.4f}  "
        f"val_loss={val_loss:.4f}  "
        f"val_acc={val_acc*100:.2f}%"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "ecgformer_best.pth")
        print(f"  ✓ Saved best model → ecgformer_best.pth")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

ECGformer | trainable params: 402,434
ECGformer | trainable params: 402,434


/tmp/ipykernel_350062/2874418297.py:21: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  scheduler.step(val_loss)


Epoch [  1/100] train_loss=0.5966  val_loss=0.4175  val_acc=80.79%
  ✓ Saved best model → ecgformer_best.pth
Epoch [  2/100] train_loss=0.3770  val_loss=0.3371  val_acc=90.00%
  ✓ Saved best model → ecgformer_best.pth
Epoch [  3/100] train_loss=0.3247  val_loss=0.3098  val_acc=87.05%
  ✓ Saved best model → ecgformer_best.pth
Epoch [  4/100] train_loss=0.2949  val_loss=0.2733  val_acc=90.79%
  ✓ Saved best model → ecgformer_best.pth
Epoch [  5/100] train_loss=0.2720  val_loss=0.2565  val_acc=93.10%
  ✓ Saved best model → ecgformer_best.pth
Epoch [  6/100] train_loss=0.2559  val_loss=0.2424  val_acc=92.13%
  ✓ Saved best model → ecgformer_best.pth
Epoch [  7/100] train_loss=0.2429  val_loss=0.2322  val_acc=91.98%
  ✓ Saved best model → ecgformer_best.pth
Epoch [  8/100] train_loss=0.2295  val_loss=0.2224  val_acc=94.11%
  ✓ Saved best model → ecgformer_best.pth
Epoch [  9/100] train_loss=0.2201  val_loss=0.2131  val_acc=92.04%
  ✓ Saved best model → ecgformer_best.pth
Epoch [ 10/100] tra

In [15]:
# load the model and evaluate on test set precision, recall, f1-score for each class
model.load_state_dict(torch.load("ecgformer_best.pth"))
model.eval()
from sklearn.metrics import classification_report
all_preds = []
all_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to("cuda"), y_batch.to("cuda")
        outputs = model(X_batch)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())
print(classification_report(all_labels, all_preds, target_names=le.classes_))

              precision    recall  f1-score   support

           N       1.00      0.98      0.99     14842
           O       0.88      0.97      0.92      2207

    accuracy                           0.98     17049
   macro avg       0.94      0.97      0.95     17049
weighted avg       0.98      0.98      0.98     17049

